# Cross-generation semantic-count comparison (aigen vs nogen vs plain)

Compares **feedback / generation types against each other** on raw semantic-tag counts
(`objects`, `stuff`, `scene_category`, `spatial_relations`, `attr_color`, `adjectives`).
This is the semantic-count complement to `cross_gen_similarity_comparison.ipynb`, and the
cross-generation counterpart to `aigen_semantic_counts.ipynb` (which compares *tasks* within
one generation). Here the comparison axis is **generation**.

Source: `<processed_dir>/nlp_analysis/trials_final_semantic_tags.csv` for each condition.
This is NOT a `config.FILENAMES` canonical table, so it is resolved by hand (like
`aigen_semantic_counts.ipynb`). Conditions without the file yet are skipped automatically:
add generations to `GENS` as their data lands — `plain` is forward-compatible and needs no
code change.

Each tag column is a stringified list; the count is its length.

**Raw counts only** — these are not yet validated against the image, and no GPT baseline is
included yet.

**Conventions** (identical to `cross_gen_similarity_comparison.ipynb`)
- One fixed, colorblind-safe color per **generation** (Dark2), reused in every figure.
- Two views: (1) counts over attempts, faceted by task; (2) improvement delta (attempt 3 - 1).

In [1]:
import sys
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Find project root (the dir containing config.py) regardless of where the notebook sits.
project_root = Path.cwd()
while not (project_root / "config.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
print("project root:", project_root)

project root: /mnt/hdd/anatkorol/Imagination_in_translation


In [2]:
# --- Which generations to compare. Add "plain" once its semantic-tags CSVs exist. ---
GENS = ["aigen", "nogen"]
SEM_FILE = "trials_final_semantic_tags.csv"

# Semantic-tag categories: csv column -> nice axis label. Each cell is a stringified list.
CATEGORIES = {
    "objects": "Objects",
    "stuff": "Stuff",
    "scene_category": "Scene category",
    "spatial_relations": "Spatial relations",
    "attr_color": "Color attributes",
    "adjectives": "Adjectives",
}


def _count(cell):
    """Length of a stringified list cell; 0 for empty / NaN / unparseable."""
    if pd.isna(cell):
        return 0
    try:
        val = ast.literal_eval(cell)
    except (ValueError, SyntaxError):
        return 0
    return len(val) if isinstance(val, (list, tuple)) else 0


# The semantic-tags CSV lives in a nlp_analysis subfolder of each condition's processed_dir
# and is not a config.load canonical table, so we resolve + tag it by hand here.
slugs = [c for g in GENS for c in config.GROUPS_BY_GEN[g]]
frames = []
for c in slugs:
    p = config.paths_for(c).processed_dir / "nlp_analysis" / SEM_FILE
    if not p.exists():
        print(f"skip {c}: {SEM_FILE} not found ({p})")
        continue
    d = pd.read_csv(p)
    cm = config.mapping_data["CONDITIONS"][c]
    d["condition"], d["generation"], d["task"] = c, cm["generation"], cm["task"]
    for col in CATEGORIES:
        d[f"n_{col}"] = d[col].apply(_count)
    frames.append(d)
    print(f"loaded {c}: {len(d)} rows, {d['uid'].nunique()} participants")

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
assert not df.empty, f"no {SEM_FILE} found for any of {slugs}"
df["attempt"] = df["attempt"].astype(int)

# Keep only generations that actually loaded (skips any without data yet).
GEN_ORDER = [g for g in ["aigen", "nogen", "plain"] if g in set(df["generation"])]

GRAPHS = config.ROOT / "analysis" / "outputs" / config.DATASET / "combined" / "graphs" / "nlp_graphs"
GRAPHS.mkdir(parents=True, exist_ok=True)
print("generations loaded:", GEN_ORDER, "-> saving figures to:", GRAPHS)
df.groupby(["generation", "task"]).size().unstack("task")

loaded aigen_perc: 15 rows, 2 participants
loaded aigen_imm: 15 rows, 1 participants
loaded aigen_del: 7 rows, 1 participants
loaded nogen_perc: 327 rows, 23 participants
loaded nogen_imm: 309 rows, 22 participants
loaded nogen_del: 213 rows, 17 participants
generations loaded: ['aigen', 'nogen'] -> saving figures to: /mnt/hdd/anatkorol/Imagination_in_translation/analysis/outputs/Full_experiment/combined/graphs/nlp_graphs


task,delay,immediate,perception
generation,,,
aigen,7,15,15
nogen,213,309,327


In [ ]:
# --- Style + fixed color maps (per generation, reused in every figure) ---
sns.set_theme(style="whitegrid")

TASK_ORDER  = ["perception", "immediate", "delay"]
TASK_LABELS = {"perception": "Perception", "immediate": "Immediate memory", "delay": "Delayed memory"}
ATTEMPTS    = [1, 2, 3]

# Dark2: green / orange / purple - one fixed color per generation (matches the similarity nb).
GEN_COLORS = {"aigen": "#1b9e77", "nogen": "#d95f02", "plain": "#7570b3"}
GEN_LABELS = {"aigen": "AI-gen (image)", "nogen": "No-gen (text)", "plain": "Plain (baseline)"}

In [ ]:
# --- Reusable helpers -------------------------------------------------------
def _ppt_attempt_means(category):
    """Mean count per (generation, task, uid, attempt) - the per-participant points,
    collapsing across that participant's sessions / ground-truth images."""
    col = f"n_{category}"
    return (df.groupby(["generation", "task", "uid", "attempt"], as_index=False)[col]
              .mean()
              .rename(columns={col: "count"}))


def plot_generations_over_attempts_by_task(category, save=True):
    """VIEW 1: one panel per task; within each panel one line per generation
    (mean +/- 95% bootstrap CI) over attempts 1-3."""
    label = CATEGORIES[category]
    g = _ppt_attempt_means(category)
    fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(6.0 * len(TASK_ORDER), 5.5),
                             sharey=True)
    axes = np.atleast_1d(axes)
    for ax, task in zip(axes, TASK_ORDER):
        for gen in GEN_ORDER:
            sub = g[(g["task"] == task) & (g["generation"] == gen)]
            if sub.empty:
                continue
            sns.lineplot(data=sub, x="attempt", y="count", ax=ax, color=GEN_COLORS[gen],
                         lw=2.6, marker="o", markersize=7, errorbar=("ci", 95),
                         label=GEN_LABELS[gen])
        ax.set_title(TASK_LABELS[task])
        ax.set_xticks(ATTEMPTS); ax.set_xticklabels([f"Attempt {a}" for a in ATTEMPTS])
        ax.set_xlabel(""); ax.set_ylabel(f"Mean {label.lower()} count (per participant)")
        ax.legend(title="Generation", frameon=True, fontsize=8)
    fig.suptitle(f"{label}: count over attempts - generations compared by task")
    fig.tight_layout()
    if save:
        out = GRAPHS / f"crossgen_{category}_count_over_attempts_by_task.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print("saved:", out)
    plt.show()

## View 1 - generations over attempts, faceted by task
For each category: one panel per task; within a panel, one line per generation (mean +/- 95% CI).
Shows whether the feedback type changes how many tags of each kind people write, holding task fixed.

In [ ]:
for category in CATEGORIES:
    plot_generations_over_attempts_by_task(category)

## View 2 - improvement delta (attempt 3 - attempt 1) by condition
How much does the count change across the feedback loop, and does that depend on the feedback /
generation type? Per participant we take the mean count at attempt 3 minus attempt 1
(positive = more tags by the last attempt). Bars = mean +/- SE, x = task, color = generation,
faint points = individual participants; the dashed line marks no change.

In [ ]:
def plot_improvement_delta_bars(category, save=True):
    """Improvement delta = per-participant mean(attempt 3) - mean(attempt 1), one bar per
    (generation x task). x = task, hue/dodge = generation, faint per-participant points overlaid."""
    label = CATEGORIES[category]
    pm = _ppt_attempt_means(category)
    wide = pm.pivot_table(index=["generation", "task", "uid"], columns="attempt", values="count")
    if not {1, 3}.issubset(wide.columns):
        print(f"{category}: missing attempt 1 or 3 - skipping")
        return
    wide = wide.dropna(subset=[1, 3])
    d = (wide[3] - wide[1]).rename("delta").reset_index()

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    sns.barplot(data=d, x="task", y="delta", hue="generation",
                order=TASK_ORDER, hue_order=GEN_ORDER,
                palette=GEN_COLORS, errorbar="se", capsize=0.06, ax=ax)
    sns.stripplot(data=d, x="task", y="delta", hue="generation",
                  order=TASK_ORDER, hue_order=GEN_ORDER,
                  palette=GEN_COLORS, dodge=True, size=3, alpha=0.35,
                  linewidth=0, ax=ax, legend=False)
    ax.axhline(0, color="black", ls="--", lw=1)
    ax.set_xlabel(""); ax.set_ylabel(f"delta {label.lower()} count\n(attempt 3 - attempt 1)")
    ax.set_xticks(range(len(TASK_ORDER)))
    ax.set_xticklabels([TASK_LABELS[t] for t in TASK_ORDER], rotation=15)
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles[:len(GEN_ORDER)], [GEN_LABELS[g] for g in GEN_ORDER],
              title="Generation", frameon=True)
    ax.set_title(f"Change in {label.lower()} count across attempts (participant means +/- SE)")
    fig.tight_layout()
    if save:
        out = GRAPHS / f"crossgen_{category}_count_improvement_delta.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print("saved:", out)
    plt.show()


for category in CATEGORIES:
    plot_improvement_delta_bars(category)

---
### Adding `plain` later
Once `plain`'s `trials_final_semantic_tags.csv` files exist, add `"plain"` to `GENS` (cell 3)
and re-run. `GEN_COLORS` / `GEN_LABELS` already include it, and `GEN_ORDER` picks up any
generation that actually loaded, so no plotting code changes are needed. The same holds for
any nogen task condition (imm / del) whose CSV lands later.